# Model Routing for VSS (Switchyard)

Routes the VSS agent's LLM traffic through a model router
([NVIDIA-NeMo/Switchyard](https://github.com/NVIDIA-NeMo/Switchyard), Apache-2.0)
instead of a single fixed endpoint. VSS already supports remote
OpenAI-compatible LLM endpoints, so **no service is added to any VSS compose
file and no VSS profile is edited**. This notebook:

1. builds Switchyard from a pinned source ref,
2. runs it as a local container and verifies real requests route across targets,
3. composes (but does **not** deploy) a VSS build that points `LLM_BASE_URL` at
   the router.

Routing is **disabled by default**: the composed build lands in a build
directory and nothing about a running or future VSS deployment changes unless
you deploy that build yourself in step 5.


## 1. Settings

### 1.1 Initialize upstream provider variables

Run this once, then set values in **one** option of 1.2.


In [ ]:
# ================== Upstream provider vars (set by ONE of (a)/(b) in 1.2) ==================
NVIDIA_API_KEY = UPSTREAM_BASE_URL = UPSTREAM_API_KEY = ""


### 1.2 Choose ONE upstream the router forwards to

The router holds the upstream credential; VSS never sees it.

#### (a) NVIDIA inference hub (default)


In [ ]:
# (a) NVIDIA inference hub. Prefer exporting NVIDIA_API_KEY in the environment
# before starting Jupyter; a value pasted here lands in the saved notebook.
UPSTREAM_BASE_URL = "https://inference-api.nvidia.com/v1"
NVIDIA_API_KEY = ""  # empty: read from the environment (preferred)


#### (b) Any OpenAI-compatible endpoint (self-hosted or other cloud)

In [ ]:
# (b) OpenAI-compatible endpoint. Prefer exporting UPSTREAM_API_KEY in the
# environment; a value pasted here lands in the saved notebook.
# UPSTREAM_BASE_URL = ""  # e.g. "http://my-endpoint:8000/v1"
# UPSTREAM_API_KEY  = ""  # empty: read from the environment (preferred)


### 1.3 Advanced settings (defaults; usually leave alone)

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

# ================== Default model-routing settings ==================
# Switchyard source pin (v0.2.0). Do not track `main`; record what you ran
# so a routed evaluation can be repeated.
SWITCHYARD_REPO_URL = "https://github.com/NVIDIA-NeMo/Switchyard.git"
SWITCHYARD_REF = "cef42319255e6d6788fd6287fedde0a0f115065e"
# Router runtime. The router is unauthenticated and forwards with the
# operator's upstream credential. The default "local" publishes it on
# loopback plus the Docker bridge gateway: reachable from this host and its
# containers, unreachable from the outside network. Set an explicit address
# (or "0.0.0.0") only deliberately, on a network you trust to reach it.
ROUTER_PORT = "4000"
ROUTER_BIND = "local"
ROUTER_CONTAINER = "vss-model-router"
# Upstream wire format: "openai_chat" or "anthropic_messages". The router
# translates its client API to this, so VSS talks to it the same way either way.
UPSTREAM_FORMAT = "openai_chat"
# Two-tier target set. VSS's LLM_NAME selects a *route id*, not a model.
ROUTER_TARGET_CAPABLE = "openai/openai/gpt-5.6-sol"
ROUTER_TARGET_EFFICIENT = "openai/openai/gpt-5.6-luna"
ROUTER_ROUTE = "switchyard/random"
# Which VSS Foundation the repoint is composed against (offline, not deployed).
VSS_FOUNDATION_PROFILE = "dev-profile-lvs"
# Both off by default: the notebook proves the path and deploys nothing.
MODEL_ROUTING_APPLY = "false"   # "true": print the exact deploy command (still not run)
ROUTER_TEARDOWN = "false"       # "true": remove the router container at the end (CI does)

# ================== Derived (no need to touch) ==================
HOME_DIR = Path.home().resolve()
VSS_REPO_DIR = Path(os.environ.get("VSS_REPO_DIR", Path.cwd())).resolve()
if not (VSS_REPO_DIR / "deploy" / "docker" / "compose.yml").is_file():
    # Notebook lives in deploy/docker/scripts/; walk up when run in place.
    for parent in Path.cwd().resolve().parents:
        if (parent / "deploy" / "docker" / "compose.yml").is_file():
            VSS_REPO_DIR = parent
            break
WORK_DIR = Path(os.environ.get("MODEL_ROUTING_WORK_DIR", HOME_DIR / "vss-model-routing")).resolve()
BUILD_DIR = VSS_REPO_DIR / "_builds" / "model-routing"
SWITCHYARD_DIR = WORK_DIR / "switchyard"
ROUTER_CONFIG = WORK_DIR / "config.toml"
ROUTER_IMAGE = f"switchyard:{SWITCHYARD_REF[:12]}"
ROUTER_PORT = int(str(ROUTER_PORT))
ROUTER_URL = f"http://127.0.0.1:{ROUTER_PORT}"

NVIDIA_API_KEY = (NVIDIA_API_KEY or os.environ.get("NVIDIA_API_KEY", "")).strip()
UPSTREAM_API_KEY = (UPSTREAM_API_KEY or os.environ.get("UPSTREAM_API_KEY", "")
                    or NVIDIA_API_KEY).strip()
_truthy = ("1", "true", "yes", "on")
MODEL_ROUTING_APPLY = str(MODEL_ROUTING_APPLY).strip().lower() in _truthy
ROUTER_TEARDOWN = str(ROUTER_TEARDOWN).strip().lower() in _truthy

FOUNDATION_ENV = (VSS_REPO_DIR / "deploy" / "docker" / "developer-profiles"
                  / VSS_FOUNDATION_PROFILE / "overrides.env")
ROOT_COMPOSE = VSS_REPO_DIR / "deploy" / "docker" / "compose.yml"
# The lvs profile ships /path/to placeholders for these two; real values are
# required for `docker compose config` to resolve.
os.environ["VSS_APPS_DIR"] = str(VSS_REPO_DIR / "deploy" / "docker")
os.environ.setdefault("VSS_DATA_DIR", str(WORK_DIR / "vss-apps-data"))

WORK_DIR.mkdir(parents=True, exist_ok=True)
print(f"MODEL_ROUTING_UPSTREAM: {UPSTREAM_BASE_URL}")
print(f"MODEL_ROUTING_ROUTE: {ROUTER_ROUTE}")
print(f"Switchyard ref: {SWITCHYARD_REF}")
print(f"Work dir: {WORK_DIR}")
print(f"Build dir: {BUILD_DIR}")


## 2. Preflight

In [ ]:
import socket

RED, GREEN, YELLOW, RESET = "\033[31m", "\033[32m", "\033[33m", "\033[0m"


def port_is_free(port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(("127.0.0.1", port)) != 0


def router_already_ours() -> bool:
    r = subprocess.run(["docker", "ps", "--filter", f"name=^{ROUTER_CONTAINER}$",
                        "--format", "{{.Names}}"], capture_output=True, text=True)
    return r.stdout.strip() == ROUTER_CONTAINER


required_checks = {
    "Upstream endpoint set": bool(UPSTREAM_BASE_URL),
    "Upstream API key set": bool(UPSTREAM_API_KEY),
    "SWITCHYARD_REF pinned": bool(SWITCHYARD_REF),
    "VSS repo layout (deploy/docker/compose.yml)": ROOT_COMPOSE.is_file(),
    f"Foundation profile ({VSS_FOUNDATION_PROFILE})": FOUNDATION_ENV.is_file(),
    "docker": shutil.which("docker") is not None,
    "docker compose": subprocess.run(["docker", "compose", "version"],
                                     capture_output=True).returncode == 0,
    "git": shutil.which("git") is not None,
    "curl": shutil.which("curl") is not None,
    f"port {ROUTER_PORT} free (or held by {ROUTER_CONTAINER})":
        port_is_free(ROUTER_PORT) or router_already_ours(),
}

failed = [name for name, ok in required_checks.items() if not ok]
for name, ok in required_checks.items():
    print(f"  {GREEN + 'ok ' if ok else RED + 'MISSING'}{RESET}  {name}")
if failed:
    raise RuntimeError(f"Preflight failed: {', '.join(failed)}")
print(f"{GREEN}Preflight passed.{RESET}")


## 3. Build and run the router

### 3.1 Fetch and build Switchyard (pinned)

Builds from the repository's own `Dockerfile`: a `debian:bookworm-slim`
runtime with only the `switchyard-server` binary and CA certificates, running
as a non-root user on port 4000. **No image is pinned or pulled from a
registry**; the pin is a source ref. Safe to re-run: an existing checkout is
reset to the pin and the Docker layer cache makes rebuilds cheap.


In [ ]:
def run(cmd, check=True, echo=True, **kw):
    print("$", " ".join(map(str, cmd)))
    r = subprocess.run([str(c) for c in cmd], capture_output=True, text=True, **kw)
    if echo and r.stdout.strip():
        print(r.stdout.strip()[-2000:])
    if r.returncode != 0:
        print(r.stderr.strip()[-2000:])
        if check:
            raise RuntimeError(f"command failed ({r.returncode}): {cmd[0]}")
    return r


if not (SWITCHYARD_DIR / ".git").is_dir():
    run(["git", "clone", "--filter=blob:none", SWITCHYARD_REPO_URL, SWITCHYARD_DIR])
run(["git", "-C", SWITCHYARD_DIR, "fetch", "origin", SWITCHYARD_REF], echo=False)
# -f: a rerun against a dirty checkout must still land exactly on the pin.
run(["git", "-C", SWITCHYARD_DIR, "checkout", "-f", "--detach", SWITCHYARD_REF], echo=False)
head = run(["git", "-C", SWITCHYARD_DIR, "rev-parse", "HEAD"], echo=False).stdout.strip()
assert head == SWITCHYARD_REF, f"checkout drifted: {head}"
print(f"Switchyard at {head}")

run(["docker", "build", "-t", ROUTER_IMAGE, SWITCHYARD_DIR], echo=False)
print(f"Built {ROUTER_IMAGE}")


### 3.2 Write the router config

The schema belongs to Switchyard and moves with its releases. Two targets,
two routes: `switchyard/random` splits traffic ~50/50, `switchyard/control`
passes everything to the efficient target. Run the same work through both;
the difference is what routing bought.


In [ ]:
ROUTER_CONFIG.write_text(f'''\
# Written by deploy_model_routing.ipynb; verified against Switchyard v0.2.0.
schema_version = 1

[llm_clients.upstream]
format = "{UPSTREAM_FORMAT}"
base_url = "{UPSTREAM_BASE_URL}"
# The router holds the upstream credential, not VSS.
api_key_env = "UPSTREAM_API_KEY"

[targets.capable]
id = "{ROUTER_TARGET_CAPABLE}"
llm_client = "upstream"

[targets.efficient]
id = "{ROUTER_TARGET_EFFICIENT}"
llm_client = "upstream"

[routes.random]
id = "switchyard/random"
type = "random"
targets = ["capable", "efficient"]

[routes.control]
id = "switchyard/control"
type = "passthrough"
target = "efficient"
''')
print(f"Wrote {ROUTER_CONFIG}")


### 3.3 Start the router

The container gets the upstream key through its environment and writes a
routing decision log to the work directory. By default it is published on
loopback and the Docker bridge gateway only: the router is unauthenticated,
so anything that can reach the port can spend the upstream credential.


In [ ]:
import json as _json
import time
import urllib.request

gw = run(["docker", "network", "inspect", "bridge", "--format",
          "{{(index .IPAM.Config 0).Gateway}}"], check=False, echo=False).stdout.strip()
DOCKER_BRIDGE_GW = gw or "172.17.0.1"
if ROUTER_BIND == "local":
    publish = ["-p", f"127.0.0.1:{ROUTER_PORT}:4000",
               "-p", f"{DOCKER_BRIDGE_GW}:{ROUTER_PORT}:4000"]
else:
    publish = ["-p", f"{ROUTER_BIND}:{ROUTER_PORT}:4000"]

run(["docker", "rm", "-f", ROUTER_CONTAINER], check=False, echo=False)
# The server runs as a non-root user, so the mounted log directory must be
# world-writable. The decision log is truncated so a rerun starts clean.
log_dir = WORK_DIR / "logs"
log_dir.mkdir(exist_ok=True)
log_dir.chmod(0o777)
(log_dir / "routing.jsonl").write_text("")
(log_dir / "routing.jsonl").chmod(0o666)

# The upstream key goes to the container through a 0600 env file so it
# never appears on a command line or in notebook output.
env_file = WORK_DIR / "router.env"
env_file.touch()
env_file.chmod(0o600)
env_file.write_text(f"UPSTREAM_API_KEY={UPSTREAM_API_KEY}\n")
run(["docker", "run", "-d", "--name", ROUTER_CONTAINER,
     *publish,
     "-v", f"{ROUTER_CONFIG}:/etc/switchyard/config.toml:ro",
     "-v", f"{WORK_DIR / 'logs'}:/var/log/switchyard",
     "--env-file", env_file,
     ROUTER_IMAGE,
     "--config", "/etc/switchyard/config.toml", "--port", "4000",
     "--routing-log-file", "/var/log/switchyard/routing.jsonl"], echo=False)
# The container copied its environment at start; the file has done its job.
env_file.unlink()

deadline = time.time() + 60
while True:
    try:
        with urllib.request.urlopen(f"{ROUTER_URL}/v1/stats", timeout=5) as r:
            _json.loads(r.read())
        break
    except Exception:
        if time.time() > deadline:
            run(["docker", "logs", "--tail", "40", ROUTER_CONTAINER], check=False)
            raise RuntimeError("router did not become healthy within 60s")
        time.sleep(2)
print(f"Router healthy at {ROUTER_URL}")


## 4. Verify routing end to end

Real requests through the router to the real upstream. Seeing both targets
served is the check that routing, not just proxying, happened; over 12
requests a single-target split has ~0.05% odds.


In [ ]:
N_PROBES = 12
payload = _json.dumps({
    "model": ROUTER_ROUTE,
    "messages": [{"role": "user", "content": "Reply with the single word: ok"}],
    "max_tokens": 8,
}).encode()

ok = 0
for _ in range(N_PROBES):
    req = urllib.request.Request(
        f"{ROUTER_URL}/v1/chat/completions", data=payload, method="POST",
        headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=120) as r:
            body = _json.loads(r.read())
        if body.get("choices"):
            ok += 1
    except Exception as exc:
        print(f"probe failed: {type(exc).__name__}: {exc}")

with urllib.request.urlopen(f"{ROUTER_URL}/v1/stats", timeout=10) as r:
    stats = _json.loads(r.read())
print(_json.dumps(stats, indent=2)[:1200])

routing_log = WORK_DIR / "logs" / "routing.jsonl"
decisions = [_json.loads(line) for line in
             routing_log.read_text().splitlines() if line.strip()] \
             if routing_log.is_file() else []
# v0.2.0 decision-log schema: one line per request, `model` = routed target id.
served = {}
for d in decisions:
    target = str(d.get("model") or "?")
    served[target] = served.get(target, 0) + 1

if ok < N_PROBES:
    raise RuntimeError(f"only {ok}/{N_PROBES} probes succeeded")
expected_targets = {ROUTER_TARGET_CAPABLE, ROUTER_TARGET_EFFICIENT}
if set(served) != expected_targets:
    raise RuntimeError(
        f"expected exactly the configured targets {sorted(expected_targets)} "
        f"served, got: {served or 'no decisions logged'}")
print(f"ROUTER_VERIFIED: {ok}/{N_PROBES} probes ok, served split {served}")


## 5. Compose the VSS repoint (not deployed; disabled by default)

This is a configuration of an endpoint VSS already supports:

| Variable | Value | Why |
|---|---|---|
| `LLM_MODE` | `remote` | existing supported override set |
| `LLM_NAME_SLUG` | `none` | with remote mode, resolves the local LLM NIM out of `COMPOSE_PROFILES`, freeing its GPU |
| `LLM_BASE_URL` | router address, **no trailing `/v1`** | the agent appends `/v1` itself |
| `LLM_NAME` | a route id (`switchyard/random`) | with a router, this selects a routing policy, not a model |

Values are rewritten **in place**: Compose resolves dotenv references against
entries parsed *so far*, so moving a key after its consumers silently breaks
paths like `VST_CONFIG_PATH`. The checked-in profile is never edited.


In [ ]:
import re

BUILD_DIR.mkdir(parents=True, exist_ok=True)
override_env = BUILD_DIR / "override.env"


def host_routable_ip() -> str:
    # The host address a bridge-networked container can reach. A short
    # hostname is not resolvable from inside the containers; the primary
    # outbound IP is. No packet is sent by this probe.
    with socket.socket(socket.AF_INET, socket.SOCK_DGRAM) as s:
        s.connect(("8.8.8.8", 80))
        return s.getsockname()[0]


# The URL must match where the router listens. In "local" mode that is
# the Docker bridge gateway, which the VSS containers can reach.
if os.environ.get("MODEL_ROUTING_VSS_URL"):
    router_for_vss = os.environ["MODEL_ROUTING_VSS_URL"]
elif ROUTER_BIND == "local":
    router_for_vss = f"http://{DOCKER_BRIDGE_GW}:{ROUTER_PORT}"
elif ROUTER_BIND == "0.0.0.0":
    router_for_vss = f"http://{host_routable_ip()}:{ROUTER_PORT}"
else:
    router_for_vss = f"http://{ROUTER_BIND}:{ROUTER_PORT}"

text = FOUNDATION_ENV.read_text()
for key, value in (("LLM_MODE", "remote"), ("LLM_NAME_SLUG", "none"),
                   ("LLM_BASE_URL", router_for_vss), ("LLM_NAME", ROUTER_ROUTE)):
    text, n = re.subn(rf"^{key}=.*$", f"{key}={value}", text, flags=re.MULTILINE)
    if n != 1:
        raise RuntimeError(f"expected exactly one {key}= line in {FOUNDATION_ENV}, found {n}")
override_env.write_text(text)

# Validate against the checked-in root compose; its include paths are
# relative to its own directory, so it cannot be copied elsewhere.
# Compose interpolates process-environment secrets into its output, so every
# config call runs with the secrets scrubbed: resolved.yml is a deployable
# artifact, not a credential store.
scrubbed_env = {**os.environ}
for secret in ("NVIDIA_API_KEY", "UPSTREAM_API_KEY", "NGC_CLI_API_KEY",
               "OPENAI_API_KEY", "ANTHROPIC_API_KEY"):
    scrubbed_env.pop(secret, None)
run(["docker", "compose", "--env-file", override_env,
     "-f", ROOT_COMPOSE, "config", "--quiet"], echo=False, env=scrubbed_env)


def service_set(env_file):
    r = run(["docker", "compose", "--env-file", env_file, "-f", ROOT_COMPOSE,
             "config", "--services"], echo=False, env=scrubbed_env)
    return set(r.stdout.split())


foundation, routed = service_set(FOUNDATION_ENV), service_set(override_env)
added, removed = sorted(routed - foundation), sorted(foundation - routed)
if added:
    raise RuntimeError(f"routing must add no service, added: {added}")
print(f"Services: {len(foundation)} -> {len(routed)}; added none; "
      f"resolved away local LLM: {removed}")

resolved = run(["docker", "compose", "--env-file", override_env,
                "-f", ROOT_COMPOSE, "config"], echo=False, env=scrubbed_env)
for secret_value in (NVIDIA_API_KEY, UPSTREAM_API_KEY):
    if secret_value and secret_value in resolved.stdout:
        raise RuntimeError("a credential reached the resolved compose output")
(BUILD_DIR / "resolved.yml").write_text(resolved.stdout)
print(f"VSS_ROUTING_COMPOSE: valid ({BUILD_DIR})")

if MODEL_ROUTING_APPLY:
    print("\nTo deploy the routed build (this notebook does not):")
    print(f"  docker compose --env-file {override_env} -f {ROOT_COMPOSE} up -d")
else:
    print("\nRouting stays disabled: nothing about a running VSS deployment changed.")


## 6. Teardown (optional)

Off by default so an interactive session can keep using the router. CI sets
`ROUTER_TEARDOWN=true`.


In [ ]:
if ROUTER_TEARDOWN:
    # check=True: "done" must mean the container is actually gone.
    run(["docker", "rm", "-f", ROUTER_CONTAINER], echo=False)
    print("ROUTER_TEARDOWN: done")
else:
    print(f"ROUTER_TEARDOWN: skipped; router still serving at {ROUTER_URL}")
